# CNN baseline backbone + DQN：OASIS1-only grouped DE 调参，支持 5-fold / 5-seed

这个 notebook 只使用 OASIS1，只读取 OASIS1 manifest。

它调用的是：

```bash
python train_cnn_dqn_de.py --run-de --cnn-arch ...
```

可以把 classifier backbone 切到：`current`、`basic_cnn`、`resnet18`、`densenet121`。

关键设置：

- `DATASET_TO_RUN = 'oasis1'` 固定为 OASIS1。
- `CNN_ARCH = 'current' / 'basic_cnn' / 'resnet18' / 'densenet121'` 控制 CNN backbone。
- `DE_EVAL_MODE = 'kfold'` 时，每个 DE candidate 用 `--de-eval-mode kfold --de-eval-folds 5` 评分。
- `DE_EVAL_MODE = 'seeds'` 时，每个 DE candidate 用 `--de-eval-mode seeds --de-eval-seeds 5` 评分。
- `DE_EVAL_MODE = 'split'` 是旧逻辑：每个 candidate 只用一个 inner split。
- DQN / final training 都用 `--best-checkpoint-metric composite`，最终结果来自 best checkpoint。


In [ ]:
from pathlib import Path
import subprocess, sys, json, shlex
import pandas as pd

PROJECT_DIR = Path.cwd()
SCRIPT = PROJECT_DIR / 'train_cnn_dqn_de.py'
BUILD_MANIFEST = PROJECT_DIR / 'build_manifest.py'

MANIFEST_ROOT = PROJECT_DIR / 'outputs_cv_manifests_al'
OUTPUT_ROOT = PROJECT_DIR / 'outputs_de_grouped_current_dqn'
CACHE_ROOT = PROJECT_DIR / 'cache_de_grouped_current_dqn'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# OASIS1-only：只使用 al_manifest.csv。
MANIFESTS = {
    'oasis1': MANIFEST_ROOT / 'oasis1' / 'manifests' / 'al_manifest.csv',
}

def run_cmd(cmd, cwd=PROJECT_DIR):
    cmd = [str(x) for x in cmd]
    print('\n$ ' + ' '.join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, check=True, cwd=str(cwd))

print('SCRIPT =', SCRIPT, '  # DE / DQN 主入口')
assert SCRIPT.exists(), f'Missing training script: {SCRIPT}'


## 1. 基础设置

最重要的是 `CNN_ARCH` 和 `DE_EVAL_MODE`：

```python
CNN_ARCH = 'current'      # 原 compact 2.5D CNN
CNN_ARCH = 'basic_cnn'    # BasicCNN baseline
CNN_ARCH = 'resnet18'     # ResNet18 baseline
CNN_ARCH = 'densenet121'  # DenseNet121 baseline

DE_EVAL_MODE = 'kfold'    # 5-fold 调参
DE_EVAL_MODE = 'seeds'    # 5-seed 调参
DE_EVAL_MODE = 'split'    # 旧版单 split 调参
```


In [ ]:
DATASET_TO_RUN = 'oasis1'   # OASIS1-only
DEVICE = 'cuda'
USE_AMP = True
SEED = 42
BEST_CHECKPOINT_METRIC = 'composite'

# 换 CNN backbone 就改这里：'current' / 'basic_cnn' / 'resnet18' / 'densenet121'
CNN_ARCH = 'current'
# resnet18/densenet121 默认使用 ImageNet 预训练；如果你的环境不能下载/加载权重，改成 False。
PRETRAINED_BACKBONE = True

IMAGE_SIZE = (128, 128)

# 输入模式：
#   '2.5d' = sagittal + coronal + axial 多切片拼成多通道输入，支持 DQN/AL 主实验
#   '2d'   = 单平面 2D 多切片，每个 MRI 的 N_SLICES 张 slice 预测取平均，更接近 train_all_2d1.py 的 baseline
INPUT_MODE = '2.5d'  # '2.5d' / '2d'

# 2.5D 参数：input_channels = 3 * SLICES_PER_PLANE
SLICES_PER_PLANE = 10

# 2D 参数：只在 INPUT_MODE='2d' 时使用
PLANE = 'axial'      # 'axial' / 'coronal' / 'sagittal'
N_SLICES = 15

SLICE_FRACTION_MIN = 0.15
SLICE_FRACTION_MAX = 0.85

# 这里改调参方式：'kfold' / 'seeds' / 'split'
DE_EVAL_MODE = 'kfold'
DE_EVAL_FOLDS = 5
DE_EVAL_SEEDS = 5

# DE 分组：cnn_train / cnn_arch / al_train / dqn_explore / dqn_replay / scope / joint / all
DE_GROUP = 'cnn_train'
DE_FITNESS_MODE = 'al'      # classifier 或 al；DQN 相关组必须用 al 才有意义
DE_ONLY = False

DE_POP_SIZE = 16
DE_GENERATIONS = 8
DE_F = 0.7
DE_CR = 0.7
DE_SUBSET = 240
DE_AL_CYCLES = 5
DE_AL_BUDGET = 32
DE_AL_MAX_ANNOTATIONS = -1

# 固定主动学习协议，DE 不调这些，保证 candidate 公平
INITIAL_LABEL_FRACTION = 0.10
CYCLES = 5
BUDGET_PER_CYCLE = 32
MAX_ANNOTATIONS = -1

# 用 al_manifest 的固定 split 时，外层 folds=1 会严格使用 train/val/test/unlabeled_pool。
# 注意：DE_EVAL_MODE='kfold' 是 DE candidate 内部评估方式，不等于最终训练 folds。
OUTER_FOLDS_FOR_DE_SEARCH = 1
RESPECT_MANIFEST_SPLITS = True


# Heatmap 外部调用：最终训练后可以单独跑，不需要重训。
RUN_FINAL_HEATMAPS_ONLY = False
HEATMAP_CHECKPOINT = 'best'
HEATMAP_METHOD = 'both'          # 'input-gradient' / 'branch-gradcam' / 'both'
HEATMAP_TARGET = 'pred'          # 'pred' / 'true' / 'demented'
HEATMAP_ALPHA = 0.45
HEATMAP_DPI = 180
HEATMAP_SAVE_NPY = False
HEATMAP_SAVE_ALL_SLICE_PNGS = False
HEATMAP_DISPLAY_FRACTIONS = (0.5, 0.5, 0.5)
HEATMAP_DISPLAY_INDICES = None

FIXED_BEST_PARAMS = {
    # 'lr': 0.001,
    # 'dropout': 0.4,
}



## 2. 工具函数


In [ ]:
SANITY_BOUNDS = {
    'lr': (5e-5, 3e-3),
    'initial_epochs': (30, 120),
    'retrain_epochs': (3, 40),
    'dropout': (0.0, 0.7),
    'focal_gamma': (0.0, 3.0),
    'weight_decay': (0.0, 1e-3),
    'epsilon_start': (0.2, 1.0),
    'epsilon_final': (0.0, 0.2),
    'epsilon_decay': (0.97, 0.9999),
    'label_cost': (0.05, 1.0),
    'dqn_lr': (1e-5, 5e-3),
    'scope_weight': (0.0, 0.1),
    'q_policy_temperature': (0.25, 4.0),
    'advantage_clip': (2.0, 10.0),
}
INT_KEYS = {'batch_size', 'initial_epochs', 'retrain_epochs', 'base_filters', 'cnn_stacks', 'dense_units',
            'q_hidden_dim', 'q_mlp_layers', 'replay_size', 'replay_batch_size', 'dqn_updates_per_cycle'}

def sanitize_params(params: dict) -> dict:
    out = dict(params)
    for k, (lo, hi) in SANITY_BOUNDS.items():
        if k in out:
            v = float(out[k])
            v2 = min(max(v, lo), hi)
            if v2 != v:
                print(f'[sanity] clamp {k}: {v} -> {v2}')
            out[k] = v2
    for k in INT_KEYS:
        if k in out:
            out[k] = int(round(float(out[k])))
    if 'epsilon_start' in out and 'epsilon_final' in out and float(out['epsilon_final']) > float(out['epsilon_start']):
        out['epsilon_final'] = float(out['epsilon_start'])
    # 兼容旧别名
    if 'scope_alpha' in out:
        out['scope_gamma'] = out['scope_alpha']
    if 'scope_weight' in out:
        out['scope_coef'] = out['scope_weight']
    return out

def params_to_cli_auto(params: dict):
    out = []
    for k, v in params.items():
        cli = '--' + k.replace('_', '-')
        out.extend([cli, str(v)])
    return out

def find_best_params_file(out_dir: Path) -> Path:
    preferred = out_dir / 'fold_01' / 'de_search' / 'de_best_params.json'
    if preferred.exists():
        return preferred
    hits = sorted(out_dir.rglob('de_best_params.json'))
    if not hits:
        raise FileNotFoundError(f'No de_best_params.json found under {out_dir}')
    return hits[0]


## 3. 自动连续跑多个 DE 分组

这个 cell 的命令里明确包含：

```bash
--run-de --de-only --de-eval-mode kfold --de-eval-folds 5
```

或当 `DE_EVAL_MODE='seeds'` 时包含：

```bash
--run-de --de-only --de-eval-mode seeds --de-eval-seeds 5
```


In [ ]:
AUTO_RUN_GROUPS = [
    'cnn_train',
    'cnn_arch',
    'al_train',
    'dqn_explore',
    'dqn_replay',
    'scope',
    'joint',
]

AUTO_GROUP_FITNESS_MODE = {
    'cnn_train': 'classifier',
    'cnn_arch': 'classifier',
    'al_train': 'al',
    'dqn_explore': 'al',
    'dqn_replay': 'al',
    'scope': 'al',
    'joint': 'al',
}

if CNN_ARCH != 'current' and 'cnn_arch' in AUTO_RUN_GROUPS:
    AUTO_RUN_GROUPS = [g for g in AUTO_RUN_GROUPS if g != 'cnn_arch']
    print(f"[info] CNN_ARCH={CNN_ARCH}: skip DE group 'cnn_arch' because baseline architecture is fixed.")

AUTO_FIXED_PARAMS = sanitize_params(dict(FIXED_BEST_PARAMS))


def run_de_group_auto(group_name: str, fixed_params: dict) -> dict:
    manifest_csv = MANIFESTS[DATASET_TO_RUN]
    assert manifest_csv.exists(), f'Manifest not found: {manifest_csv}'

    fitness_mode = AUTO_GROUP_FITNESS_MODE.get(group_name, 'al')
    cache_npz = CACHE_ROOT / f'{DATASET_TO_RUN}_{INPUT_MODE}_{PLANE}_img{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}_2d{N_SLICES}_25d{SLICES_PER_PLANE}_{SLICE_FRACTION_MIN}_{SLICE_FRACTION_MAX}.npz'
    out_dir = OUTPUT_ROOT / f'{DATASET_TO_RUN}_{INPUT_MODE}_{CNN_ARCH}_de_{group_name}_{fitness_mode}_{DE_EVAL_MODE}'

    cmd = [
        sys.executable, SCRIPT,
        '--manifest-csv', manifest_csv,
        '--output-dir', out_dir,
        '--cache-npz', cache_npz,
        '--image-size', str(IMAGE_SIZE[0]), str(IMAGE_SIZE[1]),
        '--input-mode', INPUT_MODE,
        '--plane', PLANE,
        '--n-slices', str(N_SLICES),
        '--slices-per-plane', str(SLICES_PER_PLANE),
        '--slice-fraction-min', str(SLICE_FRACTION_MIN),
        '--slice-fraction-max', str(SLICE_FRACTION_MAX),
        '--device', DEVICE,
        '--seed', str(SEED),
        '--cnn-arch', CNN_ARCH,
        '--folds', str(OUTER_FOLDS_FOR_DE_SEARCH),
        '--best-checkpoint-metric', BEST_CHECKPOINT_METRIC,
        '--initial-label-fraction', str(INITIAL_LABEL_FRACTION),
        '--cycles', str(CYCLES),
        '--budget-per-cycle', str(BUDGET_PER_CYCLE),
        '--max-annotations', str(MAX_ANNOTATIONS),
        '--run-de',
        '--de-only',
        '--de-group', group_name,
        '--de-fitness-mode', fitness_mode,
        '--de-pop-size', str(DE_POP_SIZE),
        '--de-generations', str(DE_GENERATIONS),
        '--de-F', str(DE_F),
        '--de-CR', str(DE_CR),
        '--de-subset', str(DE_SUBSET),
        '--de-eval-mode', DE_EVAL_MODE,
        '--de-eval-folds', str(DE_EVAL_FOLDS),
        '--de-eval-seeds', str(DE_EVAL_SEEDS),
        '--de-al-cycles', str(DE_AL_CYCLES),
        '--de-al-budget', str(DE_AL_BUDGET),
        '--de-al-max-annotations', str(DE_AL_MAX_ANNOTATIONS),
    ]

    if CNN_ARCH in {'resnet18', 'densenet121'}:
        cmd.append('--pretrained-backbone' if PRETRAINED_BACKBONE else '--no-pretrained-backbone')

    if OUTER_FOLDS_FOR_DE_SEARCH == 1:
        cmd.append('--respect-manifest-splits' if RESPECT_MANIFEST_SPLITS else '--no-respect-manifest-splits')
    if USE_AMP:
        cmd.append('--amp')

    fixed_params = sanitize_params(fixed_params)
    cmd += params_to_cli_auto(fixed_params)

    print('\n' + '#' * 90)
    print(f'Running group={group_name}, fitness_mode={fitness_mode}, de_eval_mode={DE_EVAL_MODE}')
    print('Fixed params passed to this group:')
    print(json.dumps(fixed_params, indent=2, ensure_ascii=False))
    print('#' * 90 + '\n')

    run_cmd(cmd)

    best_path = find_best_params_file(out_dir)
    result = json.loads(best_path.read_text())
    group_best = sanitize_params(result.get('best_params', result))

    print('\n' + '=' * 80)
    print(f'Best params for group: {group_name} | fitness_mode={fitness_mode} | de_eval_mode={DE_EVAL_MODE}')
    print('best_path =', best_path)
    print(json.dumps(group_best, indent=2, ensure_ascii=False))
    print('=' * 80 + '\n')
    return group_best

RUN_AUTO_DE = True
if RUN_AUTO_DE:
    for group in AUTO_RUN_GROUPS:
        print(f'\n\n######## Running DE group: {group} ########')
        group_best = run_de_group_auto(group, AUTO_FIXED_PARAMS)
        AUTO_FIXED_PARAMS.update(group_best)
        AUTO_FIXED_PARAMS = sanitize_params(AUTO_FIXED_PARAMS)

    auto_best_path = OUTPUT_ROOT / f'{DATASET_TO_RUN}_{CNN_ARCH}_auto_grouped_best_params_{DE_EVAL_MODE}.json'
    auto_best_path.write_text(json.dumps(AUTO_FIXED_PARAMS, indent=2, ensure_ascii=False))
    print('\nFinal accumulated params:')
    print(json.dumps(AUTO_FIXED_PARAMS, indent=2, ensure_ascii=False))
    print('Saved to:', auto_best_path)

FINAL_PARAMS = dict(globals().get('AUTO_FIXED_PARAMS', FIXED_BEST_PARAMS))


## 4. 用 best params 跑最终训练

这个 cell 不再运行 DE，只把上一步搜索出的 `FINAL_PARAMS` 作为普通 CLI 参数传给训练脚本。

最终训练依然使用 best checkpoint：`--best-checkpoint-metric composite`。


In [ ]:
RUN_FINAL = True
FINAL_FOLDS = 5  # 最终汇报想做 5-fold CV 就保留 5；想严格用 al_manifest 的固定 split 就改成 1。
FINAL_RESPECT_MANIFEST_SPLITS = (FINAL_FOLDS == 1)

FINAL_PARAMS = sanitize_params(dict(globals().get('AUTO_FIXED_PARAMS', FIXED_BEST_PARAMS)))

if RUN_FINAL:
    manifest_csv = MANIFESTS[DATASET_TO_RUN]
    assert manifest_csv.exists(), f'Manifest not found: {manifest_csv}'
    cache_npz = CACHE_ROOT / f'{DATASET_TO_RUN}_{INPUT_MODE}_{PLANE}_img{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}_2d{N_SLICES}_25d{SLICES_PER_PLANE}_{SLICE_FRACTION_MIN}_{SLICE_FRACTION_MAX}.npz'

    final_out = OUTPUT_ROOT / f'{DATASET_TO_RUN}_{INPUT_MODE}_{CNN_ARCH}_final_fixed_params_{DE_EVAL_MODE}'
    cmd = [
        sys.executable, SCRIPT,
        '--manifest-csv', manifest_csv,
        '--output-dir', final_out,
        '--cache-npz', cache_npz,
        '--image-size', str(IMAGE_SIZE[0]), str(IMAGE_SIZE[1]),
        '--input-mode', INPUT_MODE,
        '--plane', PLANE,
        '--n-slices', str(N_SLICES),
        '--slices-per-plane', str(SLICES_PER_PLANE),
        '--slice-fraction-min', str(SLICE_FRACTION_MIN),
        '--slice-fraction-max', str(SLICE_FRACTION_MAX),
        '--device', DEVICE,
        '--seed', str(SEED),
        '--cnn-arch', CNN_ARCH,
        '--folds', str(FINAL_FOLDS),
        '--best-checkpoint-metric', BEST_CHECKPOINT_METRIC,
        '--initial-label-fraction', str(INITIAL_LABEL_FRACTION),
        '--cycles', str(CYCLES),
        '--budget-per-cycle', str(BUDGET_PER_CYCLE),
        '--max-annotations', str(MAX_ANNOTATIONS),
    ]
    if CNN_ARCH in {'resnet18', 'densenet121'}:
        cmd.append('--pretrained-backbone' if PRETRAINED_BACKBONE else '--no-pretrained-backbone')

    if FINAL_FOLDS == 1:
        cmd.append('--respect-manifest-splits' if FINAL_RESPECT_MANIFEST_SPLITS else '--no-respect-manifest-splits')
    if USE_AMP:
        cmd.append('--amp')
    cmd += params_to_cli_auto(FINAL_PARAMS)
    run_cmd(cmd)


## 5. 外部生成最终模型 heatmap（不重新训练）

这个 cell 调用：

```bash
python train_cnn_dqn_de.py --heatmap-only --heatmap-checkpoint best ...
```

它会读取最终训练输出目录里的 `config.json`、每个 `fold_XX/classifier_best.pt` 和 test predictions。


In [ ]:
if RUN_FINAL_HEATMAPS_ONLY:
    manifest_csv = MANIFESTS[DATASET_TO_RUN]
    cache_npz = CACHE_ROOT / f'{DATASET_TO_RUN}_{INPUT_MODE}_{PLANE}_img{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}_2d{N_SLICES}_25d{SLICES_PER_PLANE}_{SLICE_FRACTION_MIN}_{SLICE_FRACTION_MAX}.npz'
    final_out = OUTPUT_ROOT / f'{DATASET_TO_RUN}_{INPUT_MODE}_{CNN_ARCH}_final_fixed_params_{DE_EVAL_MODE}'
    assert final_out.exists(), f'Final training output dir not found: {final_out}'

    cmd = [
        sys.executable, SCRIPT,
        '--manifest-csv', manifest_csv,
        '--output-dir', final_out,
        '--cache-npz', cache_npz,
        '--input-mode', INPUT_MODE,
        '--plane', PLANE,
        '--n-slices', str(N_SLICES),
        '--heatmap-only',
        '--heatmap-checkpoint', HEATMAP_CHECKPOINT,
        '--heatmap-method', HEATMAP_METHOD,
        '--heatmap-target', HEATMAP_TARGET,
        '--heatmap-alpha', str(HEATMAP_ALPHA),
        '--heatmap-dpi', str(HEATMAP_DPI),
        '--heatmap-display-fractions', *[str(x) for x in HEATMAP_DISPLAY_FRACTIONS],
        '--device', DEVICE,
    ]
    if HEATMAP_DISPLAY_INDICES is not None:
        cmd += ['--heatmap-display-indices', *[str(x) for x in HEATMAP_DISPLAY_INDICES]]
    if HEATMAP_SAVE_NPY:
        cmd.append('--heatmap-save-npy')
    if HEATMAP_SAVE_ALL_SLICE_PNGS:
        cmd.append('--heatmap-save-all-slice-pngs')
    run_cmd(cmd)



## 5. 可选：单命令“先 DE 后训练”

如果不想自动分组，只想一次性搜索一组参数并立刻训练，用下面这个 cell。它调用同一个主脚本，但使用：

```bash
--run-de --apply-de-best --de-eval-mode kfold --de-eval-folds 5
```

默认不运行。


In [ ]:
RUN_SINGLE_SHOT_DE_AND_TRAIN = False

if RUN_SINGLE_SHOT_DE_AND_TRAIN:
    manifest_csv = MANIFESTS[DATASET_TO_RUN]
    cache_npz = CACHE_ROOT / f'{DATASET_TO_RUN}_{INPUT_MODE}_{PLANE}_single_shot_img{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}_2d{N_SLICES}_25d{SLICES_PER_PLANE}.npz'
    out_dir = OUTPUT_ROOT / f'{DATASET_TO_RUN}_{INPUT_MODE}_{CNN_ARCH}_single_shot_de_{DE_GROUP}_{DE_EVAL_MODE}'
    cmd = [
        sys.executable, SCRIPT,
        '--manifest-csv', manifest_csv,
        '--output-dir', out_dir,
        '--cache-npz', cache_npz,
        '--image-size', str(IMAGE_SIZE[0]), str(IMAGE_SIZE[1]),
        '--input-mode', INPUT_MODE,
        '--plane', PLANE,
        '--n-slices', str(N_SLICES),
        '--slices-per-plane', str(SLICES_PER_PLANE),
        '--device', DEVICE,
        '--seed', str(SEED),
        '--cnn-arch', CNN_ARCH,
        '--folds', '1',
        '--respect-manifest-splits',
        '--best-checkpoint-metric', BEST_CHECKPOINT_METRIC,
        '--run-de',
        '--apply-de-best',
        '--de-group', DE_GROUP,
        '--de-fitness-mode', DE_FITNESS_MODE,
        '--de-pop-size', str(DE_POP_SIZE),
        '--de-generations', str(DE_GENERATIONS),
        '--de-eval-mode', DE_EVAL_MODE,
        '--de-eval-folds', str(DE_EVAL_FOLDS),
        '--de-eval-seeds', str(DE_EVAL_SEEDS),
        '--initial-label-fraction', str(INITIAL_LABEL_FRACTION),
        '--cycles', str(CYCLES),
        '--budget-per-cycle', str(BUDGET_PER_CYCLE),
    ]
    if CNN_ARCH in {'resnet18', 'densenet121'}:
        cmd.append('--pretrained-backbone' if PRETRAINED_BACKBONE else '--no-pretrained-backbone')
    if USE_AMP:
        cmd.append('--amp')
    run_cmd(cmd)
